# 🎬 LongCat Studio SE — Talking Avatar

**Ladda upp ett foto + skriv text → Få en talande video**

Baserat på [LongCat-Video-Avatar 1.5](https://github.com/meituan-longcat/LongCat-Video) av Meituan (MIT-licens).

---

## ⚠️ GPU-krav
| GPU | VRAM | Status |
|-----|------|--------|
| **A100 (40GB)** | 40 GB | ✅ Rekommenderas |
| **V100 (32GB)** | 32 GB | ✅ Fungerar |
| T4 (15GB) | 15 GB | ❌ För liten |

> **Aktivera A100:** Runtime → Change runtime type → A100 *(kräver Colab Pro, ~10 USD/mån)*

---

## Workflow
1. Kör **Steg 1–7** en gång (tar ~15 min första gången, sedan < 2 min via Drive-cache)
2. Kör **Steg 8** för att starta Gradio-studion
3. Ladda upp bild + skriv svensk text → klicka Generera → ladda ner video

In [ ]:
# ── Steg 1: GPU-check ──────────────────────────────────────────────────
import torch, subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:800])

if not torch.cuda.is_available():
    raise RuntimeError("❌ Ingen GPU hittad! Aktivera GPU: Runtime → Change runtime type")

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"\nGPU: {gpu_name}")
print(f"VRAM: {gpu_mem_gb:.1f} GB")

if gpu_mem_gb < 30:
    print("\n❌ VARNING: Behöver minst 32 GB VRAM (A100/V100).")
    print("   Gå till: Runtime → Change runtime type → A100")
    raise RuntimeError("Otillräcklig VRAM")
else:
    print("\n✅ GPU OK — tillräcklig VRAM för LongCat Avatar 1.5")

In [ ]:
# ── Steg 2: Systempaket ────────────────────────────────────────────────
import subprocess, sys

def run(cmd, desc=""):
    print(f"▶ {desc or cmd[:60]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  STDERR: {r.stderr[-300:]}")
    else:
        print("  ✅ OK")
    return r.returncode

run("apt-get install -qq ffmpeg", "ffmpeg")
print("✅ Systempaket klara")

In [ ]:
# ── Steg 3: Python-paket ────────────────────────────────────────────────
# Installeras i rätt ordning (flash-attn kräver ninja + packaging first)

pkgs_base = [
    ("ninja psutil packaging",            "Byggverktyg"),
    ("huggingface_hub[cli]",               "HuggingFace CLI"),
    ("edge-tts nest_asyncio",              "Svensk TTS (gratis, offline-ej-nödvändig)"),
    ("soundfile librosa",                  "Audiobearbetning"),
    ("gradio>=4.0",                        "Gradio UI"),
    ("audio-separator",                     "Röstseparation"),
]

for pkg, desc in pkgs_base:
    run(f"pip install -q {pkg}", desc)

# flash-attn — försöker installera, faller tillbaka på sdpa om det misslyckas
print("▶ flash-attn (kan ta 2-3 min att kompilera...)")
rc = run(
    "pip install -q flash_attn==2.7.4.post1 --no-build-isolation 2>/dev/null",
    "flash-attn"
)
FLASH_ATTN_OK = (rc == 0)
print(f"  flash-attn: {'✅' if FLASH_ATTN_OK else '⚠️ ej tillgänglig — patchar config till sdpa'}")

print("\n✅ Python-paket klara")

In [ ]:
# ── Steg 4: Klona LongCat ──────────────────────────────────────────────
import os, subprocess

LONGCAT_DIR = "/content/LongCat-Video"

if not os.path.exists(LONGCAT_DIR):
    print("▶ Klonar LongCat-Video...")
    subprocess.run(
        "git clone -q --single-branch --branch main "
        "https://github.com/meituan-longcat/LongCat-Video "
        f"{LONGCAT_DIR}",
        shell=True, check=True
    )
    print("  ✅ Klonad")
else:
    print("  ✅ LongCat-Video finns redan")

os.chdir(LONGCAT_DIR)

print("▶ Installerar LongCat requirements...")
subprocess.run("pip install -q -r requirements.txt",      shell=True, check=True)
subprocess.run("pip install -q -r requirements_avatar.txt", shell=True, check=True)
print("✅ LongCat installerat — katalog:", os.getcwd())

In [ ]:
# ── Steg 5: Montera Google Drive (modell-cache) ────────────────────────
# Modellerna (~25 GB) laddas ner EN gång och sparas i Drive.
# Nästa session: startar direkt utan nedladdning.

from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_MODELS = "/content/drive/MyDrive/LongCat-Models"
os.makedirs(DRIVE_MODELS, exist_ok=True)

WEIGHTS_DIR = f"{LONGCAT_DIR}/weights"

# Symlänk från LongCat/weights → Drive
if not os.path.exists(WEIGHTS_DIR):
    os.symlink(DRIVE_MODELS, WEIGHTS_DIR)
    print(f"✅ weights/ → {DRIVE_MODELS}")
else:
    print(f"✅ weights/ finns redan")

print(f"Modeller sparas i: {DRIVE_MODELS}")

In [ ]:
# ── Steg 6: Ladda ner modeller ─────────────────────────────────────────
# Första gången: ~25 GB, tar 10-20 min beroende på nätverkshastighet.
# Efterföljande sessioner: direkt från Drive (< 1 min).

import os, subprocess

DRIVE_MODELS  = "/content/drive/MyDrive/LongCat-Models"
BASE_MODEL    = f"{DRIVE_MODELS}/LongCat-Video"
AVATAR_MODEL  = f"{DRIVE_MODELS}/LongCat-Video-Avatar-1.5"

def download_if_missing(repo_id, local_dir):
    marker = os.path.join(local_dir, ".download_complete")
    if os.path.exists(marker):
        print(f"  ✅ {os.path.basename(local_dir)} — hittad i Drive-cache")
        return
    print(f"  ⬇️  Laddar ner {repo_id} → {local_dir}")
    print("     (kan ta 5-15 min...)")
    subprocess.run(
        f"huggingface-cli download {repo_id} --local-dir {local_dir}",
        shell=True, check=True
    )
    open(marker, 'w').close()  # markör: nedladdning klar
    print(f"  ✅ {os.path.basename(local_dir)} klar")

print("=== Modell-nedladdning ===")
download_if_missing("meituan-longcat/LongCat-Video",           BASE_MODEL)
download_if_missing("meituan-longcat/LongCat-Video-Avatar-1.5", AVATAR_MODEL)

# Patcha config om flash-attn saknas
import json
for config_path in [
    f"{BASE_MODEL}/dit/config.json",
    f"{AVATAR_MODEL}/base_model/config.json",
    f"{AVATAR_MODEL}/base_model_int8/config.json",
]:
    if os.path.exists(config_path):
        with open(config_path) as f:
            cfg = json.load(f)
        if not FLASH_ATTN_OK and cfg.get('attn_implementation') == 'flash_attention_2':
            cfg['attn_implementation'] = 'sdpa'
            with open(config_path, 'w') as f:
                json.dump(cfg, f, indent=2)
            print(f"  Patched {os.path.basename(config_path)}: flash_attention_2 → sdpa")

print("\n✅ Alla modeller redo!")

In [ ]:
# ── Steg 7: Verifiera installation ─────────────────────────────────────
import os, torch

checks = [
    ("/content/LongCat-Video/longcat_video",             "LongCat Python-modul"),
    ("/content/LongCat-Video/weights/LongCat-Video",     "Basmodell"),
    ("/content/LongCat-Video/weights/LongCat-Video-Avatar-1.5", "Avatar 1.5 modell"),
    ("/usr/bin/ffmpeg",                                  "ffmpeg"),
]

all_ok = True
for path, name in checks:
    ok = os.path.exists(path)
    print(f"{'✅' if ok else '❌'} {name}: {path}")
    if not ok:
        all_ok = False

# Testa imports
try:
    import sys
    sys.path.insert(0, '/content/LongCat-Video')
    from longcat_video.pipeline_longcat_video_avatar import LongCatVideoAvatarPipeline
    print("✅ LongCatVideoAvatarPipeline importerad")
except Exception as e:
    print(f"❌ Import-fel: {e}")
    all_ok = False

try:
    import edge_tts
    print("✅ edge-tts importerat")
except:
    print("❌ edge-tts saknas")
    all_ok = False

if all_ok:
    print("\n🎉 INSTALLATION KOMPLETT — Kör Steg 8 för att starta studion!")
else:
    print("\n⚠️ Något saknas — kontrollera felmeddelandena ovan")

In [ ]:
# ── Steg 8: LongCat Studio — Gradio UI ────────────────────────────────
# Laddar modellen EN gång i minnet, genererar sedan snabbt per körning.

import os, sys, json, asyncio, tempfile, subprocess, shutil
import nest_asyncio
import gradio as gr
import edge_tts
import torch

nest_asyncio.apply()
sys.path.insert(0, '/content/LongCat-Video')
os.chdir('/content/LongCat-Video')

# ── Svenska Edge-TTS röster ────────────────────────────────────────────
SWEDISH_VOICES = {
    "Sofie (kvinna) — standard": "sv-SE-SofieNeural",
    "Mattias (man)": "sv-SE-MattiasNeural",
    "HilaRi (kvinna) — finlandssvenska": "sv-FI-HarriNeural",
}

STANDARD_PROMPT = (
    "A professional person is speaking clearly and naturally, "
    "making eye contact, natural expressions, high quality, realistic."
)

# ── TTS: text → WAV ───────────────────────────────────────────────────
async def _tts_async(text: str, voice: str, out_path: str):
    communicate = edge_tts.Communicate(text, voice)
    # edge-tts skriver MP3; konvertera till WAV 16kHz
    mp3_path = out_path.replace('.wav', '.mp3')
    await communicate.save(mp3_path)
    subprocess.run(
        f"ffmpeg -y -i '{mp3_path}' -ar 16000 -ac 1 '{out_path}'",
        shell=True, check=True, capture_output=True
    )
    os.remove(mp3_path)

def text_to_speech(text: str, voice_label: str) -> str:
    voice = SWEDISH_VOICES.get(voice_label, "sv-SE-SofieNeural")
    out_path = tempfile.mktemp(suffix=".wav")
    asyncio.get_event_loop().run_until_complete(_tts_async(text, voice, out_path))
    return out_path

# ── Generera video via torchrun ────────────────────────────────────────
def generate_video(
    image_path: str,
    text: str,
    voice_label: str,
    custom_audio,
    prompt: str,
    resolution: str,
    num_segments: int,
    progress=gr.Progress()
):
    if image_path is None:
        return None, "❌ Ladda upp ett porträttfoto."
    if not text.strip() and custom_audio is None:
        return None, "❌ Ange text ELLER ladda upp ett ljud-klipp."

    tmp_dir = tempfile.mkdtemp()
    out_dir = os.path.join(tmp_dir, "output")
    os.makedirs(out_dir, exist_ok=True)

    try:
        # 1. Generera eller kopiera ljud
        progress(0.1, "Genererar svenska TTS-röst...")
        if custom_audio is not None:
            audio_path = os.path.join(tmp_dir, "audio.wav")
            subprocess.run(
                f"ffmpeg -y -i '{custom_audio}' -ar 16000 -ac 1 '{audio_path}'",
                shell=True, check=True, capture_output=True
            )
        else:
            audio_path = text_to_speech(text, voice_label)

        # 2. Spara bild
        img_path = os.path.join(tmp_dir, "portrait.jpg")
        shutil.copy(image_path, img_path)

        # 3. Skapa input JSON
        input_json_path = os.path.join(tmp_dir, "input.json")
        input_data = {
            "prompt": prompt or STANDARD_PROMPT,
            "cond_audio": {"person1": audio_path},
            "cond_image": img_path
        }
        with open(input_json_path, 'w') as f:
            json.dump(input_data, f)

        # 4. Kör LongCat-inference
        progress(0.2, "Startar LongCat-inference (8 steg med distillation)...")
        checkpoint_dir = '/content/LongCat-Video/weights/LongCat-Video-Avatar-1.5'

        cmd = (
            f"torchrun --nproc_per_node=1 "
            f"run_demo_avatar_single_audio_to_video.py "
            f"--context_parallel_size=1 "
            f"--checkpoint_dir='{checkpoint_dir}' "
            f"--stage_1=ai2v "
            f"--input_json='{input_json_path}' "
            f"--num_segments={num_segments} "
            f"--resolution={resolution} "
            f"--use_distill "
            f"--model_type=avatar-v1.5 "
            f"--use_int8 "
            f"--output_dir='{out_dir}'"
        )

        print(f"\nKör: {cmd}\n")
        result = subprocess.run(cmd, shell=True, capture_output=False, text=True)

        if result.returncode != 0:
            return None, f"❌ Inference misslyckades (exit {result.returncode})"

        # 5. Hitta output-video
        progress(0.95, "Söker output-video...")
        videos = []
        for fname in os.listdir(out_dir):
            if fname.endswith('.mp4'):
                videos.append(os.path.join(out_dir, fname))

        if not videos:
            return None, "❌ Ingen video genererades — kontrollera loggarna ovan."

        # Returnera senaste/längsta video
        video_path = max(videos, key=os.path.getsize)
        progress(1.0, "Klar!")
        return video_path, f"✅ Video klar! ({os.path.basename(video_path)})"

    except Exception as e:
        return None, f"❌ Fel: {str(e)}"


# ── Gradio UI ──────────────────────────────────────────────────────────
with gr.Blocks(
    title="LongCat Studio SE",
    theme=gr.themes.Soft(primary_hue="blue"),
) as demo:

    gr.Markdown(
        """# 🎬 LongCat Studio SE
        ### Ladda upp ett porträttfoto · Skriv vad personen ska säga · Få en talande video
        *Bygger på [LongCat-Video-Avatar 1.5](https://github.com/meituan-longcat/LongCat-Video) (MIT-licens)*
        """
    )

    with gr.Row():
        # ── Vänster: Inputs ────────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### Input")

            image_input = gr.Image(
                label="Porträttfoto (JPG/PNG)",
                type="filepath",
                height=300,
            )

            with gr.Tab("Text → Röst (TTS)"):
                text_input = gr.Textbox(
                    label="Text på svenska",
                    placeholder="Hej! Jag heter Anna och jag vill berätta om hur AI kan hjälpa ditt företag...",
                    lines=4,
                )
                voice_select = gr.Dropdown(
                    choices=list(SWEDISH_VOICES.keys()),
                    value=list(SWEDISH_VOICES.keys())[0],
                    label="Svensk röst (Microsoft Edge TTS)",
                )
                gr.Markdown(
                    "*Röster från Microsoft Edge TTS — gratis, ingen API-nyckel krävs.*"
                )

            with gr.Tab("Eget ljud"):
                audio_input = gr.Audio(
                    label="Ladda upp ljud (WAV/MP3)",
                    type="filepath",
                )
                gr.Markdown("*Ladda upp valfritt tal-klipp — ersätter TTS.*")

            with gr.Accordion("Avancerade inställningar", open=False):
                prompt_input = gr.Textbox(
                    label="Video-prompt (valfri)",
                    value=STANDARD_PROMPT,
                    lines=3,
                    placeholder="Beskriv scenen: miljö, klädsel, ton..."
                )
                resolution_select = gr.Radio(
                    choices=["480p", "720p"],
                    value="480p",
                    label="Upplösning (720p kräver mer VRAM)",
                )
                segments_input = gr.Slider(
                    minimum=1, maximum=5, step=1, value=1,
                    label="Antal segment (längre video = fler segment)",
                )

            generate_btn = gr.Button(
                "🎬  Generera video",
                variant="primary",
                size="lg"
            )

        # ── Höger: Output ──────────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### Output")

            video_output = gr.Video(
                label="Genererad video",
                height=400,
            )
            status_output = gr.Textbox(
                label="Status",
                interactive=False,
                lines=2,
            )

            gr.Markdown(
                """#### Tips för bästa resultat
                - **Foto:** Neutral bakgrund, bra belysning, tydligt ansikte
                - **Text:** Naturliga meningar utan konstiga tecken
                - **Prompt:** Beskriv miljö och klädsel för mer realistisk video
                - **Ljud CFG 4–5:** Bättre läppsynkronisering (byt i källkod vid behov)
                """
            )

    # ── Exempelbilder ──────────────────────────────────────────────────
    gr.Markdown("---\n### Exempel-prompts")
    gr.Examples(
        examples=[
            [
                None,
                "Hej, jag heter Anna och jag vill berätta om hur AI kan revolutionera din verksamhet. Kontakta oss idag för en kostnadsfri demo!",
                list(SWEDISH_VOICES.keys())[0],
                None,
                "A professional woman is speaking confidently, wearing business attire, bright modern office background, natural expressions.",
                "480p",
                1,
            ],
            [
                None,
                "Välkommen till vår butik! Vi har just fått in nya produkter som jag tror du kommer att älska. Kom in och titta!",
                list(SWEDISH_VOICES.keys())[1],
                None,
                "A friendly man is speaking warmly, casual clothing, store interior background.",
                "480p",
                1,
            ],
        ],
        inputs=[
            image_input, text_input, voice_select, audio_input,
            prompt_input, resolution_select, segments_input
        ],
        label="Klicka för att fylla i exempeldata",
    )

    generate_btn.click(
        fn=generate_video,
        inputs=[
            image_input, text_input, voice_select, audio_input,
            prompt_input, resolution_select, segments_input
        ],
        outputs=[video_output, status_output],
        show_progress="full",
    )

print("🚀 Startar LongCat Studio...")
demo.launch(share=True, debug=False)